<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-22-April-16-2026/Lecture-22_QM9-Dataset-Initial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 22 - QM9 Dataset

The QM9 dataset:
- [MoleculeNet: a benchmark for molecular machine learning](https://pubs.rsc.org/en/content/articlelanding/2018/sc/c7sc02664a)
- [QM9 in  PyG (PyTorch Geometric)](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.datasets.QM9.html)



In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit lightgbm mols2grid torch_geometric

Import all basic pacakges

In [ ]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# RDkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# For progress bar
from tqdm.auto import tqdm

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor

# LightGBM
from lightgbm import LGBMRegressor, plot_importance


tqdm.pandas()

Download dataset

In [ ]:
%%bash
wget https://deepchemdata.s3.us-west-1.amazonaws.com/datasets/qm9.tar.gz
tar xvf qm9.tar.gz

In [ ]:
!ls

In [ ]:
!head qm9.sdf

dRead dataset

In [ ]:
molecules = Chem.SDMolSupplier('qm9.sdf', removeHs=False,
                                   sanitize=False)
data = pd.read_csv("qm9.sdf.csv")



In [ ]:
data

In [ ]:
molecules[100].GetProp('_Name')

In [ ]:
property_names = list(rdMolDescriptors.Properties.GetAvailableProperties())
property_getter = rdMolDescriptors.Properties(property_names)

In [ ]:
def mol2props(smi):
    props = None
    if mol:
        props = np.array(property_getter.ComputeProperties(mol))
    return props

In [ ]:
problems = []
props = []
for i, mol in enumerate(tqdm(molecules)):
  prop = mol2props(mol)
  if prop is None: problems.append(i)
  props.append(prop)

In [ ]:
indices = [i for i, x in enumerate(props) if x is None]

In [ ]:
print(indices)

In [ ]:
data[property_names]=props

In [ ]:
len(data)

In [ ]:
print("Size before:", len(data))
data.dropna()
print("Size after:", len(data))

## Train using LightGBM

In [ ]:
train, test = train_test_split(data,test_size=0.20)

In [ ]:
train_X = train[property_names]
train_y = train['']
test_X = test[property_names]
test_y = test['gap']

In [ ]:
lgbm = LGBMRegressor()
lgbm.fit(train_X, train_y)

In [ ]:
test_pred = lgbm.predict(test_X)
train_pred = lgbm.predict(train_X)

In [ ]:
plt.plot(train_y, train_pred,'.',label='Train set')
plt.plot(test_y, test_pred,'.',label='Test set')
# to get a diagonal line
diagonal_line =np.linspace(np.min(train_y),np.max(train_y),1000)
plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
# plt.axline([0, 0], slope=1, color='red', linestyle='--')
plt.legend()
plt.ylabel("Predicted Value")
plt.xlabel("Actual Value")
plt.show()